### In this Lab, we’ll understand the audio preprocessing pipeline.

In the first section, we will explore different ways to represent audio signals, including time-domain signals, FFTs, spectrograms, and MFCC-style Mel features. In the second section, we will use a keyword spotting model, convert it to TensorFlow Lite, apply full integer post-training quantization, test the quantized model, and export the model as a C/C++ array for Arduino deployment.

**Local environment version.** This notebook is edited to run in the TinyML Python environment created for this class, rather than depending on Google Colab-specific commands. It still supports recording your own voice from the laptop microphone, loading example audio, visualizing features, evaluating the quantized model, and generating the `kws.cc` file for Arduino.


# 1. Spectrograms and MFCCs



### Install required packages

The TinyML environment already installs TensorFlow, NumPy, SciPy, Matplotlib, Jupyter, and the main course dependencies. This notebook installs only the small audio-recording/audio-processing packages needed for Lab 5.


We use the following additional packages:

1. `librosa` for audio loading and resampling.
2. `sounddevice` for local microphone recording.
3. `soundfile` for audio file support used by `librosa`.


In [ ]:
# Run this cell once inside the class TinyML environment.
# In Jupyter, %pip installs packages into the currently selected kernel environment.

%pip install -q librosa sounddevice soundfile

print("Lab 5 audio packages are ready.")


### Import everything we will need

1. `IPython.display.Audio` lets us play audio clips inside the notebook.
2. `NumPy`, `SciPy`, and `Librosa` are used for signal processing and audio resampling.
3. `TensorFlow` is used for the keyword spotting model and TensorFlow Lite conversion.
4. `Matplotlib` is used for plotting waveforms, spectra, spectrograms, and Mel features.


In [ ]:
from IPython.display import Audio, display
import os
import sys
import io
import tarfile
import zipfile
import urllib.request
import subprocess
from pathlib import Path

import numpy as np
import scipy.io.wavfile
from scipy.io.wavfile import read as wav_read
from scipy import signal

import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib import cm
import pickle
import librosa
import sounddevice as sd

print("Packages imported successfully.")
print("TensorFlow version:", tf.__version__)
print("Python executable:", sys.executable)


### Define the audio importing function

The cell below defines a local microphone recorder. When you run a recording cell, press Enter when prompted, say the target word clearly, and wait until the recording finishes.

If macOS asks for microphone permission, allow Terminal, your browser, or the application used to launch Jupyter.


In [ ]:
def _normalize_to_int16(audio):
    """Convert a mono/stereo NumPy audio array to int16 mono audio."""
    audio = np.asarray(audio)

    # Convert stereo/multichannel to mono.
    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    audio = audio.astype(np.float32)

    # If audio is already in int16-like scale, normalize it safely.
    max_abs = np.max(np.abs(audio)) if audio.size else 0.0
    if max_abs == 0:
        return audio.astype(np.int16)

    if max_abs <= 1.5:
        audio = audio * 32767.0
    else:
        audio = audio / max_abs * 32767.0

    return np.clip(audio, -32768, 32767).astype(np.int16)


def get_audio(duration_seconds=1.5, sample_rate=16000):
    """
    Record audio from the local microphone.

    Parameters
    ----------
    duration_seconds : float
        Recording duration. 1.5 seconds is usually enough for one short keyword.
    sample_rate : int
        Sampling rate in Hz. The KWS model expects 16 kHz audio.

    Returns
    -------
    audio : np.ndarray
        Mono int16 audio.
    sr : int
        Sampling rate.
    """
    input(f"Press Enter, then say one keyword clearly. Recording will last {duration_seconds:.1f} seconds...")
    print("Recording...")
    recording = sd.rec(
        int(duration_seconds * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="float32",
    )
    sd.wait()
    audio = _normalize_to_int16(recording[:, 0])
    print("Recording complete.")
    display(Audio(audio, rate=sample_rate))
    return audio, sample_rate


def load_audio_file(path, target_sr=None):
    """
    Optional fallback: load a saved audio file instead of recording live audio.
    Example:
        audio, sr = load_audio_file("my_yes.wav", target_sr=16000)
    """
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return _normalize_to_int16(audio), sr


print("Local microphone recorder defined.")


In [ ]:
# Optional cleanup cell.
# Uncomment the lines below if you want to remove generated folders/files and rerun the lab from scratch.

# import shutil
# for path in ["dataset", "extract_loudest_section", "logs", "models", "tensorflow", "tensorflow-2.14.1",
#              "train", "trimmed", "audio_files.pkl", "yes_no.pkl", "speech_micro_train_2020_05_10.tgz",
#              "tensorflow-v2.14.1.zip", "kws.cc"]:
#     p = Path(path)
#     if p.is_dir():
#         shutil.rmtree(p)
#     elif p.exists():
#         p.unlink()


### Load in the Audio Samples [Optional]

### **[A] Record your own audio samples**

In the cells below, record four short audio samples using your laptop microphone. For each cell, press Enter when prompted, say a short keyword such as **yes** or **no**, and wait until the recording finishes.


In [ ]:
audio_1, sr_1 = get_audio()
print("DONE")

In [ ]:
audio_2, sr_2 = get_audio()
print("DONE")

In [ ]:
audio_3, sr_3 = get_audio()
print("DONE")

In [ ]:
audio_4, sr_4 = get_audio()
print("DONE")

### Saving custom audio files

The cell below saves your four recorded audio samples into a local pickle file named `your_audio_files.pkl`. You can find it in the same folder as this notebook.


In [ ]:
your_audio_files = {
  'audio_1': audio_1, 'sr_1': sr_1,
  'audio_2': audio_2, 'sr_2': sr_2,
  'audio_3': audio_3, 'sr_3': sr_3,
  'audio_4': audio_4, 'sr_4': sr_4,
}
with open('your_audio_files.pkl', 'wb') as fid:            ##  store python object
  pickle.dump(your_audio_files, fid)

In [ ]:
# Plot the figure (axes - 10e-5)
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)
max_val = max(np.append(np.append(np.append(audio_1,audio_2),audio_3),audio_4))   # for y-axis range
ax1.plot(audio_1)
ax1.set_title("Audio 1", {'fontsize':20, 'fontweight':'bold'})
ax1.set_ylim(-max_val, max_val)
ax2.plot(audio_2)
ax2.set_title("Audio 2", {'fontsize':20, 'fontweight':'bold'})
ax2.set_ylim(-max_val, max_val)
ax3.plot(audio_3)
ax3.set_title("Audio 3", {'fontsize':20, 'fontweight':'bold'})
ax3.set_ylim(-max_val, max_val)
ax4.plot(audio_4)
ax4.set_title("Audio 4", {'fontsize':20, 'fontweight':'bold'})
ax4.set_ylim(-max_val, max_val)
fig.set_size_inches(18,12)

In [ ]:
# compute the FFT and take the single-sided spectrum only - computes magnitude of complex number
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)

ft_audio_1 = np.abs(2*np.fft.fft(audio_1))        # numpy.abs() gives magnitude of a complex number
ft_audio_2 = np.abs(2*np.fft.fft(audio_2))
ft_audio_3 = np.abs(2*np.fft.fft(audio_3))
ft_audio_4 = np.abs(2*np.fft.fft(audio_4))

# Plot the figure
ax1.plot(ft_audio_1)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_title("Audio 1", {'fontsize':20, 'fontweight':'bold'})
ax2.plot(ft_audio_2)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_title("Audio 2", {'fontsize':20, 'fontweight':'bold'})
ax3.plot(ft_audio_3)
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.set_title("Audio 3", {'fontsize':20, 'fontweight':'bold'})
ax4.plot(ft_audio_4)
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.set_title("Audio 4", {'fontsize':20, 'fontweight':'bold'})
fig.set_size_inches(18,12)
fig.text(0.5, 0.06, 'Frequency [Hz]', {'fontsize':20, 'fontweight':'bold'}, ha='center');
fig.text(0.08, 0.5, 'Amplitude', {'fontsize':20, 'fontweight':'bold'}, va='center', rotation='vertical');

### **[B] Load in default audio samples**

We download a pickle file from GitHub that contains default audio samples. These samples are useful if you do not want to use your own recorded samples for the first visualization part.


The following cell downloads `audio_files.pkl` using Python, so it works in the local TinyML environment without needing Colab, `wget`, or Linux-specific shell commands.


In [ ]:
AUDIO_FILE_URL = "https://github.com/tinyMLx/colabs/raw/master/audio_files.pkl"
AUDIO_FILE_PATH = Path("audio_files.pkl")

if not AUDIO_FILE_PATH.exists():
    print("Downloading audio_files.pkl...")
    urllib.request.urlretrieve(AUDIO_FILE_URL, AUDIO_FILE_PATH)
else:
    print("audio_files.pkl already exists.")

print("Default audio sample file is ready:", AUDIO_FILE_PATH.resolve())


In [ ]:
fid = open('audio_files.pkl', 'rb')
audio_files = pickle.load(fid)                          # python dictionary
audio_yes_loud = audio_files['audio_yes_loud']
sr_yes_loud = audio_files['sr_yes_loud']
audio_yes_quiet = audio_files['audio_yes_quiet']
sr_yes_quiet = audio_files['sr_yes_quiet']
audio_no_loud = audio_files['audio_no_loud']
sr_no_loud = audio_files['sr_no_loud']
audio_no_quiet = audio_files['audio_no_quiet']
sr_no_quiet = audio_files['sr_no_quiet']

### Inspect audio sample

In [ ]:
print("audio_yes_loud:", audio_yes_loud)
print("type:", type(audio_yes_loud))
print("shape:", audio_yes_loud.shape)

### Listen to Loaded Audio Samples
You can hear the audio files you just loaded below. IPython gives us a widget to play audio files through a notebook.

***Note*** - the loud yes and no audio could be too loud!

In [ ]:
Audio(audio_yes_loud, rate=sr_yes_loud)

In [ ]:
Audio(audio_yes_quiet, rate=sr_yes_quiet)

In [ ]:
Audio(audio_no_loud, rate=sr_no_loud)

In [ ]:
Audio(audio_no_quiet, rate=sr_no_quiet)

### Visualize the samples
[A] We will first visualize the audio samples as **signals**.

In [ ]:
# Plot the figure (axes - 10e-5)
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)
max_val = max(np.append(np.append(np.append(audio_yes_loud,audio_yes_quiet),audio_no_loud),audio_no_quiet))   # for y-axis range
ax1.plot(audio_yes_loud)
ax1.set_title("Yes Loud", {'fontsize':20, 'fontweight':'bold'})
ax1.set_ylim(-max_val, max_val)
ax2.plot(audio_yes_quiet)
ax2.set_title("Yes Quiet", {'fontsize':20, 'fontweight':'bold'})
ax2.set_ylim(-max_val, max_val)
ax3.plot(audio_no_loud)
ax3.set_title("No Loud", {'fontsize':20, 'fontweight':'bold'})
ax3.set_ylim(-max_val, max_val)
ax4.plot(audio_no_quiet)
ax4.set_title("No Quiet", {'fontsize':20, 'fontweight':'bold'})
ax4.set_ylim(-max_val, max_val)
fig.set_size_inches(18,12)


[B] Next, we will view the Fourier Transform of the signal i.e., the **signal in the frequency domain**. We will use `numpy.fft.fft()` for this. You can read more about the API [here](https://numpy.org/doc/stable/reference/generated/numpy.fft.fft.html).



In [ ]:
# compute the FFT and take the single-sided spectrum only - computes magnitude of complex number
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)

ft_audio_yes_loud = np.abs(2*np.fft.fft(audio_yes_loud))        # numpy.abs() gives magnitude of a complex number
ft_audio_yes_quiet = np.abs(2*np.fft.fft(audio_yes_quiet))
ft_audio_no_loud = np.abs(2*np.fft.fft(audio_no_loud))
ft_audio_no_quiet = np.abs(2*np.fft.fft(audio_no_quiet))

# Plot the figure
ax1.plot(ft_audio_yes_loud)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_title("Yes Loud", {'fontsize':20, 'fontweight':'bold'})
ax2.plot(ft_audio_yes_quiet)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_title("Yes Quiet", {'fontsize':20, 'fontweight':'bold'})
ax3.plot(ft_audio_no_loud)
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.set_title("No Loud", {'fontsize':20, 'fontweight':'bold'})
ax4.plot(ft_audio_no_quiet)
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.set_title("No Quiet", {'fontsize':20, 'fontweight':'bold'})
fig.set_size_inches(18,12)
fig.text(0.5, 0.06, 'Frequency [Hz]', {'fontsize':20, 'fontweight':'bold'}, ha='center');
fig.text(0.08, 0.5, 'Amplitude', {'fontsize':20, 'fontweight':'bold'}, va='center', rotation='vertical');

[C] Next, we will visualize the audio samples as spectrograms. We will be using `tfio.audio.spectrogram()` for this. You can read more about the API [here](https://www.tensorflow.org/io/api_docs/python/tfio/audio/spectrogram).

The implementation was adapted from -

Can you see how spectrograms can help machine learning models better differentiate between audio samples?

In [ ]:
# Convert to spectrogram and display.
# This local version uses scipy.signal.spectrogram instead of tensorflow-io.

def compute_log_spectrogram(audio, sr, nperseg=512, noverlap=384):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    f, t, Sxx = signal.spectrogram(
        audio,
        fs=sr,
        nperseg=min(nperseg, len(audio)),
        noverlap=min(noverlap, max(0, len(audio) // 2 - 1)),
        scaling="spectrum",
        mode="magnitude",
    )
    return np.log(Sxx + 1e-10)

spectrogram_yes_loud = compute_log_spectrogram(audio_yes_loud, sr_yes_loud)
spectrogram_yes_quiet = compute_log_spectrogram(audio_yes_quiet, sr_yes_quiet)
spectrogram_no_loud = compute_log_spectrogram(audio_no_loud, sr_no_loud)
spectrogram_no_quiet = compute_log_spectrogram(audio_no_quiet, sr_no_quiet)

# Plot the figure
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)
ax1.imshow(spectrogram_yes_loud, aspect='auto', origin='lower')
ax1.set_title("Yes Loud", {'fontsize':20, 'fontweight':'bold'})
ax2.imshow(spectrogram_yes_quiet, aspect='auto', origin='lower')
ax2.set_title("Yes Quiet", {'fontsize':20, 'fontweight':'bold'})
ax3.imshow(spectrogram_no_loud, aspect='auto', origin='lower')
ax3.set_title("No Loud", {'fontsize':20, 'fontweight':'bold'})
ax4.imshow(spectrogram_no_quiet, aspect='auto', origin='lower')
ax4.set_title("No Quiet", {'fontsize':20, 'fontweight':'bold'})
fig.set_size_inches(18,12)


[D] Finally, we will visualize the audio samples as **MFCCs** thereby using the Mel Scale to better associate the features to human hearing! We use the librosa library to achieve this. You can read more about the APIs we use here - [Link1](https://librosa.org/doc/main/generated/librosa.feature.melspectrogram.html), [Link2](https://librosa.org/doc/main/generated/librosa.power_to_db.html). This implementation was adapted from -


1.   https://towardsdatascience.com/getting-to-know-the-mel-spectrogram-31bca3e2d9d0




In [ ]:
# Convert to MFCC using the Mel Scale

mfcc_yes_loud = librosa.power_to_db(librosa.feature.melspectrogram(
    y=np.float32(audio_yes_loud), sr=sr_yes_loud, n_fft=2048, hop_length=512, n_mels=128), ref=np.max)
mfcc_yes_quiet = librosa.power_to_db(librosa.feature.melspectrogram(
    y=np.float32(audio_yes_quiet), sr=sr_yes_quiet, n_fft=2048, hop_length=512, n_mels=128), ref=np.max)
mfcc_no_loud = librosa.power_to_db(librosa.feature.melspectrogram(
    y=np.float32(audio_no_loud), sr=sr_no_loud, n_fft=2048, hop_length=512, n_mels=128), ref=np.max)
mfcc_no_quiet = librosa.power_to_db(librosa.feature.melspectrogram(
    y=np.float32(audio_no_quiet), sr=sr_no_quiet, n_fft=2048, hop_length=512, n_mels=128), ref=np.max)

# Plot the figure
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2)
ax1.imshow(np.swapaxes(mfcc_yes_loud, 0 ,1), interpolation='nearest', cmap=cm.viridis, origin='lower', aspect='auto')
ax1.set_title("Yes Loud", {'fontsize':20, 'fontweight':'bold'})
ax1.set_ylim(ax1.get_ylim()[::-1])
ax2.imshow(np.swapaxes(mfcc_yes_quiet, 0 ,1), interpolation='nearest', cmap=cm.viridis, origin='lower', aspect='auto')
ax2.set_title("Yes Quiet", {'fontsize':20, 'fontweight':'bold'})
ax2.set_ylim(ax2.get_ylim()[::-1])
ax3.imshow(np.swapaxes(mfcc_no_loud, 0 ,1), interpolation='nearest', cmap=cm.viridis, origin='lower', aspect='auto')
ax3.set_title("No Loud", {'fontsize':20, 'fontweight':'bold'})
ax3.set_ylim(ax3.get_ylim()[::-1])
ax4.imshow(np.swapaxes(mfcc_no_quiet, 0 ,1), interpolation='nearest', cmap=cm.viridis, origin='lower', aspect='auto')
ax4.set_title("No Quiet", {'fontsize':20, 'fontweight':'bold'})
ax4.set_ylim(ax4.get_ylim()[::-1])
fig.set_size_inches(18,12)

# 2. Keyword Spotting Model
In this section we will see how well a default pre-trained model works for the Keyword Spotting application.

This notebook uses a pre-trained 20 kB model based on [Simple Audio Recognition](https://www.tensorflow.org/tutorials/audio/simple_audio) to recognize keywords! The model is derived from a [micro_speech](https://github.com/tensorflow/tensorflow/tree/v2.4.1/tensorflow/lite/micro/examples/micro_speech) example for [TensorFlow Lite for MicroControllers](https://www.tensorflow.org/lite/microcontrollers/overview)

### Import Libraries
We import the following libraries.



In [ ]:
import tensorflow as tf
import shutil

# Download the TensorFlow source tree that contains the speech_commands helper scripts.
# We use v2.14.1 to match the pinned TensorFlow version in the class TinyML environment.

TF_SOURCE_VERSION = "v2.14.1"
TF_ZIP = Path(f"tensorflow-{TF_SOURCE_VERSION}.zip")
TF_SOURCE_DIR = Path("tensorflow")
EXTRACTED_TF_DIR = Path("tensorflow-2.14.1")
SPEECH_COMMANDS_DIR = TF_SOURCE_DIR / "tensorflow" / "examples" / "speech_commands"

if not SPEECH_COMMANDS_DIR.exists():
    if not TF_ZIP.exists():
        url = f"https://github.com/tensorflow/tensorflow/archive/refs/tags/{TF_SOURCE_VERSION}.zip"
        print("Downloading TensorFlow source:", url)
        urllib.request.urlretrieve(url, TF_ZIP)

    print("Extracting TensorFlow source. This may take a minute...")
    with zipfile.ZipFile(TF_ZIP, "r") as zf:
        zf.extractall(".")

    if TF_SOURCE_DIR.exists() and not SPEECH_COMMANDS_DIR.exists():
        shutil.rmtree(TF_SOURCE_DIR)

    if EXTRACTED_TF_DIR.exists():
        EXTRACTED_TF_DIR.rename(TF_SOURCE_DIR)

if not SPEECH_COMMANDS_DIR.exists():
    raise FileNotFoundError(
        "Could not find tensorflow/examples/speech_commands after downloading TensorFlow source."
    )

print("Speech commands helper directory:", SPEECH_COMMANDS_DIR.resolve())


In [ ]:
import sys
# Add this path so we can import the TensorFlow speech command helper modules.
sys.path.insert(0, str(SPEECH_COMMANDS_DIR.resolve()))

import input_data
import models
import numpy as np
import pickle

print("Speech command helper modules imported.")


The TensorFlow source folder contains the speech command preprocessing and model helper code used in this lab. In this local version, the source is downloaded with Python and stored in the notebook folder under `tensorflow/`.


### Configure Defaults
In the below cell, we define the words we want to train our model on. We define a comma-delimited list of the words you want to train for. All the other words you do not select will be used to train an "unknown" label so that the model does not just recognize speech but your specific words. Audio data with no spoken words will be used to train a "silence" label.

In [ ]:
WANTED_WORDS = "yes,no"

# Print the configuration to confirm it
print("Spotting these words: %s" % WANTED_WORDS)

We will use the default configurations to use the pre-trained model. **DO NOT MODIFY** the following constants as they include filepaths used in this notebook and data that is shared during training and inference.

In [ ]:
# Calculate the percentage of 'silence' and 'unknown' training samples required
# to ensure that we have equal number of samples for each label.

number_of_labels = WANTED_WORDS.count(',') + 1                               # count() counts the number of commas (substr provided)
number_of_total_labels = number_of_labels + 2                                # for 'silence' and 'unknown' label
equal_percentage_of_training_samples = int(100.0/(number_of_total_labels))
SILENT_PERCENTAGE = equal_percentage_of_training_samples
UNKNOWN_PERCENTAGE = equal_percentage_of_training_samples


# Constants which are shared during training and inference
PREPROCESS = 'micro'
WINDOW_STRIDE = 20
MODEL_ARCHITECTURE = 'tiny_conv'


# Constants for training directories and filepaths
DATASET_DIR =  'dataset/'
LOGS_DIR = 'logs/'
TRAIN_DIR = 'train/' # for training checkpoints and other files.


# Constants for inference directories and filepaths
import os
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
  os.mkdir(MODELS_DIR)
MODEL_TF = os.path.join(MODELS_DIR, 'model.pb')
MODEL_TFLITE = os.path.join(MODELS_DIR, 'model.tflite')
FLOAT_MODEL_TFLITE = os.path.join(MODELS_DIR, 'float_model.tflite')
MODEL_TFLITE_MICRO = os.path.join(MODELS_DIR, 'model.cc')
SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model')


# Constants for Quantization
QUANT_INPUT_MIN = 0.0
QUANT_INPUT_MAX = 26.0
QUANT_INPUT_RANGE = QUANT_INPUT_MAX - QUANT_INPUT_MIN


# Constants for audio process during Quantization and Evaluation
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30.0
FEATURE_BIN_COUNT = 40
BACKGROUND_FREQUENCY = 0.8
BACKGROUND_VOLUME_RANGE = 0.1
TIME_SHIFT_MS = 100.0


# URL for the dataset and train/val/test split
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
VALIDATION_PERCENTAGE = 10
TESTING_PERCENTAGE = 10

### Loading the pre-trained model

These commands will download a pre-trained model checkpoint file (the output from training) that we can use to build a model. You can read more about saving model checkpoints and using it for subsequent model inference here - [Link1](https://www.tensorflow.org/tutorials/keras/save_and_load), [Link2](https://www.tensorflow.org/guide/checkpoint)

You should see a train/ folder on the left tab. It contains the checkpoint and metadata.  

In [ ]:
PRETRAINED_MODEL_URL = "https://storage.googleapis.com/download.tensorflow.org/models/tflite/speech_micro_train_2020_05_10.tgz"
PRETRAINED_MODEL_ARCHIVE = Path("speech_micro_train_2020_05_10.tgz")

if not PRETRAINED_MODEL_ARCHIVE.exists():
    print("Downloading pretrained keyword spotting checkpoint...")
    urllib.request.urlretrieve(PRETRAINED_MODEL_URL, PRETRAINED_MODEL_ARCHIVE)
else:
    print("Pretrained checkpoint archive already exists.")

print("Extracting pretrained checkpoint...")
with tarfile.open(PRETRAINED_MODEL_ARCHIVE, "r:gz") as tar:
    tar.extractall(".")

TOTAL_STEPS = 15000  # used to identify which checkpoint file
print("Pretrained checkpoint is ready.")


### Generate a TensorFlow Model for Inference

Below, we combine relevant training results (graph, weights, etc) into a single file for inference. This process is known as **freezing** a model and the resulting model is known as a frozen model/graph, as it cannot be further re-trained after this process.

We run the freeze.py script to achieve this.

After running this cell, you should find the saved model under models/ in the left tab.

In [ ]:
# Remove any previous SavedModel and regenerate it from the pretrained checkpoint.
import shutil

if Path(SAVED_MODEL).exists():
    shutil.rmtree(SAVED_MODEL)

freeze_script = SPEECH_COMMANDS_DIR / "freeze.py"

cmd = [
    sys.executable,
    str(freeze_script),
    f"--wanted_words={WANTED_WORDS}",
    f"--window_stride_ms={WINDOW_STRIDE}",
    f"--preprocess={PREPROCESS}",
    f"--model_architecture={MODEL_ARCHITECTURE}",
    f"--start_checkpoint={TRAIN_DIR}{MODEL_ARCHITECTURE}.ckpt-{TOTAL_STEPS}",
    "--save_format=saved_model",
    f"--output_file={SAVED_MODEL}",
]

print("Running freeze.py...")
subprocess.run(cmd, check=True)
print("SavedModel generated at:", Path(SAVED_MODEL).resolve())


### Generate a TensorFlow Lite Model

We convert the frozen graph into a TensorFlow Lite model, which is fully quantized for use with embedded devices. The following cell will also print the model size, which should be under 20 kilobytes.



First, we download the speech commands dataset to use as a representative dataset for post-training quantization. This gives the converter realistic input feature values for integer quantization.


In [ ]:
model_settings = models.prepare_model_settings(
    len(input_data.prepare_words_list(WANTED_WORDS.split(','))),
    SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
    WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)

audio_processor = input_data.AudioProcessor(
    DATA_URL, DATASET_DIR,
    SILENT_PERCENTAGE, UNKNOWN_PERCENTAGE,
    WANTED_WORDS.split(','), VALIDATION_PERCENTAGE,
    TESTING_PERCENTAGE, model_settings, LOGS_DIR)

We create both a float TFLite model and a quantized TFLite model below. The quantized model uses full integer post-training quantization with a representative dataset.


In [ ]:
#with tf.Session() as sess:                                           #replaces the below line for use with TF1.x
with tf.compat.v1.Session() as sess:

  # float model
  float_converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
  float_tflite_model = float_converter.convert()
  float_tflite_model_size = open(FLOAT_MODEL_TFLITE, "wb").write(float_tflite_model)
  print()
  print("Float model is %d bytes" % float_tflite_model_size)

  # quantized model
  converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
  converter.optimizations = [tf.lite.Optimize.DEFAULT]
  #converter.inference_input_type = tf.lite.constants.INT8            #replaces the below line for use with TF1.x
  converter.inference_input_type = tf.compat.v1.lite.constants.INT8
  #converter.inference_output_type = tf.lite.constants.INT8           #replaces the below line for use with TF1.x
  converter.inference_output_type = tf.compat.v1.lite.constants.INT8

  def representative_dataset_gen():
    for i in range(100):
      data, _ = audio_processor.get_data(1, i*1, model_settings,
                                         BACKGROUND_FREQUENCY,
                                         BACKGROUND_VOLUME_RANGE,
                                         TIME_SHIFT_MS,
                                         'testing',
                                         sess)
      flattened_data = np.array(data.flatten(), dtype=np.float32).reshape(1, 1960)
      yield [flattened_data]

  converter.representative_dataset = representative_dataset_gen
  tflite_model = converter.convert()
  tflite_model_size = open(MODEL_TFLITE, "wb").write(tflite_model)
  print("Quantized model is %d bytes" % tflite_model_size)


### Testing the accuracy after Quantization

In the below cell, we run inference of our model on the entire test dataset.

In [ ]:
# Helper function to run inference
def run_tflite_inference_testSet(tflite_model_path, model_type="Float"):

  # extract test data
  np.random.seed(0) # set random seed for reproducible test results.
  #with tf.Session() as sess:                                                 #replaces the below line for use with TF1.x
  with tf.compat.v1.Session() as sess:
    test_data, test_labels = audio_processor.get_data(
        -1, 0, model_settings, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE,
        TIME_SHIFT_MS, 'testing', sess)
  test_data = np.expand_dims(test_data, axis=1).astype(np.float32)



  # Initialize the interpreter
  interpreter = tf.lite.Interpreter(tflite_model_path)
  interpreter.allocate_tensors()
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]


  # For quantized models, manually quantize the input data from float to integer
  if model_type == "Quantized":
    input_scale, input_zero_point = input_details["quantization"]
    test_data = test_data / input_scale + input_zero_point
    test_data = test_data.astype(input_details["dtype"])


  # Evaluate the predictions
  correct_predictions = 0
  for i in range(len(test_data)):
    interpreter.set_tensor(input_details["index"], test_data[i])
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    top_prediction = output.argmax()
    correct_predictions += (top_prediction == test_labels[i])


  print('%s model accuracy is %f%% (Number of test samples=%d)' % (
      model_type, (correct_predictions * 100) / len(test_data), len(test_data)))

In [ ]:
# Compute float model accuracy
run_tflite_inference_testSet(FLOAT_MODEL_TFLITE)

# Compute quantized model accuracy
run_tflite_inference_testSet(MODEL_TFLITE, model_type='Quantized')

# 3. Testing the model on example Audio
Now that we know the model is fairly accurate on the test set lets explore with some hand crafted examples just how accurate the model is in the real world!

### Load example audio samples

Here, we download and load the 'yes_no.pkl' file that contains a few example audio samples for 'yes' and 'no'.

In [ ]:
from IPython.display import Audio

YES_NO_URL = "https://github.com/tinyMLx/colabs/raw/master/yes_no.pkl"
YES_NO_PATH = Path("yes_no.pkl")

if not YES_NO_PATH.exists():
    print("Downloading yes_no.pkl...")
    urllib.request.urlretrieve(YES_NO_URL, YES_NO_PATH)
else:
    print("yes_no.pkl already exists.")

print("Example yes/no audio file is ready:", YES_NO_PATH.resolve())


In [ ]:
fid = open('yes_no.pkl', 'rb')
audio_files = pickle.load(fid)
yes1 = audio_files['yes1']
yes2 = audio_files['yes2']
yes3 = audio_files['yes3']
yes4 = audio_files['yes4']
no1 = audio_files['no1']
no2 = audio_files['no2']
no3 = audio_files['no3']
no4 = audio_files['no4']
sr_yes1 = audio_files['sr_yes1']
sr_yes2 = audio_files['sr_yes2']
sr_yes3 = audio_files['sr_yes3']
sr_yes4 = audio_files['sr_yes4']
sr_no1 = audio_files['sr_no1']
sr_no2 = audio_files['sr_no2']
sr_no3 = audio_files['sr_no3']
sr_no4 = audio_files['sr_no4']

In [ ]:
Audio(yes1, rate=sr_yes1)

In [ ]:
Audio(yes2, rate=sr_yes2)

In [ ]:
Audio(yes3, rate=sr_yes3)

In [ ]:
Audio(yes4, rate=sr_yes4)

In [ ]:
Audio(no1, rate=sr_no1)

In [ ]:
Audio(no2, rate=sr_no2)

In [ ]:
Audio(no3, rate=sr_no3)

In [ ]:
Audio(no4, rate=sr_no4)

### Test the model on the Example Files
We first need to import a series of packages and build the loudest section tool so that we can process audio files manually to send them to our model. These packages will also be used later for you to record your own audio to test the model!

In [ ]:
# No Colab-specific audio imports are needed in the local environment.
# We will use librosa + scipy for preprocessing and the get_audio() function defined earlier for recording.

import numpy as np
import librosa
import scipy.io.wavfile
from pathlib import Path

print("Local audio preprocessing tools are ready.")


In [ ]:
def _audio_to_float_mono(audio):
    """Convert an audio array to mono float32 in approximately [-1, 1]."""
    audio = np.asarray(audio)

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    audio = audio.astype(np.float32)
    max_abs = np.max(np.abs(audio)) if audio.size else 0.0

    if max_abs > 1.5:
        # int16-like input
        audio = audio / max_abs

    return audio


def extract_loudest_one_second(audio, sr, output_path="trimmed/custom_audio.wav", target_sr=16000):
    """
    Pure-Python replacement for the original extract_loudest_section utility.

    It resamples the audio to 16 kHz, finds the 1-second window with highest
    energy, writes that segment to a WAV file, and returns the output path.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    audio = _audio_to_float_mono(audio)

    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)

    target_len = int(target_sr * 1.0)

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))

    # Find the 1-second window with maximum energy.
    energy = np.convolve(audio ** 2, np.ones(target_len, dtype=np.float32), mode="valid")
    start = int(np.argmax(energy))
    segment = audio[start:start + target_len]

    # Normalize gently before writing to int16 WAV.
    max_abs = np.max(np.abs(segment)) if segment.size else 0.0
    if max_abs > 0:
        segment = 0.8 * segment / max_abs

    scipy.io.wavfile.write(output_path, target_sr, np.int16(segment * 32767))
    return str(output_path)


print("Pure-Python loudest 1-second extractor is ready.")


In [ ]:
# Helper function to run inference on a single input file/sample.
# This version avoids Linux-only build commands and works in the local class environment.

TF_SESS = tf.compat.v1.InteractiveSession()

def run_tflite_inference_singleFile(tflite_model_path, custom_audio, sr_custom_audio, model_type="Float"):

    # Step 1: Extract the loudest 1-second segment and save it as a 16 kHz WAV file.
    trimmed_wav_path = extract_loudest_one_second(
        custom_audio,
        sr_custom_audio,
        output_path="trimmed/custom_audio.wav",
        target_sr=SAMPLE_RATE,
    )

    # Step 2: Pass the WAV through the same speech_commands feature extraction pipeline
    # used by the TinyML KWS model.
    custom_audio_processor = input_data.AudioProcessor(
        None, None, 0, 0, '', 0, 0, model_settings, None
    )
    custom_audio_preprocessed = custom_audio_processor.get_features_for_wav(
        trimmed_wav_path,
        model_settings,
        TF_SESS,
    )

    # Step 3: Reshape the feature vector into the model input shape.
    custom_audio_input = custom_audio_preprocessed[0].flatten()
    test_data = np.reshape(custom_audio_input, (1, len(custom_audio_input))).astype(np.float32)

    # Step 4: Initialize the TFLite interpreter.
    interpreter = tf.lite.Interpreter(tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # Step 5: For quantized models, manually quantize the input data.
    if model_type == "Quantized":
        input_scale, input_zero_point = input_details["quantization"]
        test_data = test_data / input_scale + input_zero_point
        test_data = test_data.astype(input_details["dtype"])

    # Step 6: Run inference.
    interpreter.set_tensor(input_details["index"], test_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    top_prediction = output.argmax()

    # Step 7: Translate the model output index.
    if top_prediction == 2 or top_prediction == 3:
        top_prediction_str = WANTED_WORDS.split(',')[top_prediction - 2]
    elif top_prediction == 0:
        top_prediction_str = 'silence'
    else:
        top_prediction_str = 'unknown'

    print('%s model guessed the value to be %s' % (model_type, top_prediction_str))
    return top_prediction_str, output


In [ ]:
# Then test the model -- do they all work as you'd expect?
print("Testing yes1")
run_tflite_inference_singleFile(MODEL_TFLITE, yes1, sr_yes1, model_type="Quantized")
print("Testing yes2")
run_tflite_inference_singleFile(MODEL_TFLITE, yes2, sr_yes2, model_type="Quantized")
print("Testing yes3")
run_tflite_inference_singleFile(MODEL_TFLITE, yes3, sr_yes3, model_type="Quantized")
print("Testing yes4")
run_tflite_inference_singleFile(MODEL_TFLITE, yes4, sr_yes4, model_type="Quantized")
print("Testing no1")
run_tflite_inference_singleFile(MODEL_TFLITE, no1, sr_no1, model_type="Quantized")
print("Testing no2")
run_tflite_inference_singleFile(MODEL_TFLITE, no2, sr_no2, model_type="Quantized")
print("Testing no3")
run_tflite_inference_singleFile(MODEL_TFLITE, no3, sr_no3, model_type="Quantized")
print("Testing no4")
run_tflite_inference_singleFile(MODEL_TFLITE, no4, sr_no4, model_type="Quantized")

## Testing the model with your own data!

Try recording your own audio to test the model. You can experiment with different ways to say 'yes' and 'no'. Also, test the 'unknown' and 'silence' classes.

### Define the audio importing function

The local microphone recorder was already defined near the beginning of the notebook as `get_audio()`.


In [ ]:
print("Recorder available. Use custom_audio, sr_custom_audio = get_audio() in the next cell.")


### Record your own audio and test the model

Run the next cell, press Enter when prompted, say either **yes** or **no**, and wait until the recording finishes.


In [ ]:
custom_audio, sr_custom_audio = get_audio()
print("DONE")

In [ ]:
# Then test the model
run_tflite_inference_singleFile(MODEL_TFLITE, custom_audio, sr_custom_audio, model_type="Quantized")

# Generate a TensorFlow Lite for Microcontrollers Model
To convert the TensorFlow Lite quantized model into a C source file that can be loaded by TensorFlow Lite for Microcontrollers on Arduino we simply need to use the ```xxd``` tool to convert the ```.tflite``` file into a ```.cc``` file.

In [49]:
# The class environment runs on macOS/local Jupyter, so we do not use apt-get or xxd.
# The next cell exports the quantized TFLite model to a C/C++ source file using pure Python.

print("Ready to export the quantized TFLite model to kws.cc.")


Ready to export the quantized TFLite model to kws.cc.


In [50]:
MODEL_TFLITE = Path("models/model.tflite")   # Quantized INT8 TFLite model produced above
MODEL_TFLITE_MICRO = Path("kws.cc")          # C/C++ source file for Arduino
ARRAY_NAME = "g_model"

def tflite_to_c_array(tflite_path, output_path, array_name="g_model", bytes_per_line=12):
    tflite_path = Path(tflite_path)
    output_path = Path(output_path)

    model_bytes = tflite_path.read_bytes()

    with output_path.open("w") as f:
        f.write("// This file was generated from a quantized TensorFlow Lite model.\n")
        f.write("// Copy the byte array and length into micro_features_model.cpp if needed.\n\n")
        f.write("#include <cstdint>\n\n")
        f.write(f"alignas(8) const unsigned char {array_name}[] = {{\n")

        for i in range(0, len(model_bytes), bytes_per_line):
            chunk = model_bytes[i:i + bytes_per_line]
            hex_values = ", ".join(f"0x{b:02x}" for b in chunk)
            f.write(f"  {hex_values},\n")

        f.write("};\n")
        f.write(f"const int {array_name}_len = {len(model_bytes)};\n")

    return len(model_bytes)

model_size = tflite_to_c_array(MODEL_TFLITE, MODEL_TFLITE_MICRO, ARRAY_NAME)
print(f"Generated {MODEL_TFLITE_MICRO} from {MODEL_TFLITE}.")
print(f"Model size: {model_size} bytes.")
print("Use g_model and g_model_len when updating the Arduino micro_features_model.cpp file.")


Generated kws.cc from models/model.tflite.
Model size: 18984 bytes.
Use g_model and g_model_len when updating the Arduino micro_features_model.cpp file.


That's it! You've successfully converted your TensorFlow Lite model into a TensorFlow Lite for Microcontrollers-compatible C/C++ array.

In the Arduino hardware step, copy the generated `g_model` byte array and `g_model_len` value from `kws.cc` into the corresponding model source file used by the provided Arduino example.


In [51]:
# Preview the generated C/C++ model file without printing thousands of lines.
lines = MODEL_TFLITE_MICRO.read_text().splitlines()

print("\n".join(lines[:14]))
print("\n... output shortened ...\n")
print("\n".join(lines[-6:]))


// This file was generated from a quantized TensorFlow Lite model.
// Copy the byte array and length into micro_features_model.cpp if needed.

#include <cstdint>

alignas(8) const unsigned char g_model[] = {
  0x20, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x00, 0x00, 0x00, 0x00,
  0x14, 0x00, 0x20, 0x00, 0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00,
  0x0c, 0x00, 0x00, 0x00, 0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00,
  0x1c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x74, 0x00, 0x00, 0x00,
  0xd4, 0x42, 0x00, 0x00, 0xe4, 0x42, 0x00, 0x00, 0x88, 0x49, 0x00, 0x00,
  0x03, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x02, 0x00, 0x00, 0x00,
  0x34, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0xdc, 0xff, 0xff, 0xff,
  0x0e, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x13, 0x00, 0x00, 0x00,

... output shortened ...

  0x0f, 0x00, 0x00, 0x00, 0x08, 0x00, 0x04, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x03, 0x00, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x03,
  0x0c, 0x00, 0x0c, 0x00,

If you'd like to download your model for safekeeping:

1. In JupyterLab, open the file browser on the left.
2. Find `kws.cc` in the same folder as this notebook.
3. Right-click `kws.cc` and select **Download**.
